# 06 — Figures and Tables

Generates all final paper figures from locked analysis outputs.

**Figures produced:**
- `output/fig_calibration_slide3.pdf/png` — band-level calibration with diagonal
- `output/fig_calibration_zoom.pdf/png` — 0–20¢ zoom with overpriced-gap annotation
- `output/fig_exploitability.pdf/png` — exploitability forest plot by spread cost
- `output/fig_domain_calibration.pdf/png` — sports vs. political win-rate bar chart

**Reads:** `data/candidate_level_discrete.csv`, `output/buckets_overall_cw_discrete.csv`
**Writes:** four figure pairs (pdf + png)


In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils import (RNG_SEED, BOOT_REPS, FEE_RATE, FLOOR_MAX, LS_MAX, FAV_MIN, C_GRID,
                   clopper_pearson_upper,
                   C_LONGSHOT, C_FAVORITE, C_POLITICAL, C_MUTED, C_DIAGONAL,
                   FL, FT, FA)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'output')

# ── Aliases: preserve original variable names used inside each figure cell ────
C_LS    = C_LONGSHOT    # orange  — longshots (slide3 + zoom)
C_FAV   = C_FAVORITE    # teal    — favorites (slide3)
C_MUT   = C_MUTED       # grey    — floored / muted (slide3 + zoom)
C_DIA   = C_DIAGONAL    # dark    — diagonal line
C_SPORT = C_LONGSHOT    # orange  — Sports domain (domain figure)
C_POL   = C_POLITICAL   # blue    — Political domain (domain figure)
C_SIG   = C_LONGSHOT    # orange  — significant rows (forest plot)
C_NS    = C_MUTED       # grey    — non-significant rows (forest plot)
C_NULL  = C_DIAGONAL    # dark    — null reference line (forest plot)

_cp_upper = clopper_pearson_upper  # alias used by slide3 cell

cand = pd.read_csv(os.path.join(DATA_DIR, 'candidate_level_discrete.csv'))

print(f'Candidate file: {len(cand):,} rows (data/candidate_level_discrete.csv)')


Candidate file: 1,579 rows (data/candidate_level_discrete.csv)


## Exploitability and Domain Figures

Defines `_band_boot` (tuple-return) and runs exploitability computation.


In [2]:
def _band_boot(df, reps=BOOT_REPS):
    # Event-clustered bootstrap for win rate. Returns (pt, ci_lo, ci_hi).
    n     = len(df)
    n_win = int(df['won'].sum()) if n > 0 else 0
    if n == 0:
        return float('nan'), float('nan'), float('nan')
    pt = float(df['won'].mean())
    if n_win == 0:
        cp = float(scipy_stats.beta.ppf(0.95, 1, n))
        return pt, 0.0, cp
    ev_g = {ev: g['won'].values.astype(float) for ev, g in df.groupby('event')}
    evs  = list(ev_g.keys())
    boot = np.empty(reps)
    for r in range(reps):
        samp    = np.random.choice(evs, size=len(evs), replace=True)
        boot[r] = np.concatenate([ev_g[e] for e in samp]).mean()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return pt, lo, hi


np.random.seed(RNG_SEED)

# ── A. Exploitability ─────────────────────────────────────────────────────
ab = cand[(cand['snapshot_price'] > FLOOR_MAX) & (cand['snapshot_price'] <= LS_MAX)].copy()
ab['gross_pnl']  = ab['snapshot_price'] - ab['won'].astype(float)
ab['fee']        = FEE_RATE * ab['snapshot_price'] * (1.0 - ab['snapshot_price'])
ab['entry_cost'] = 1.0 - ab['snapshot_price']

ev_g_ab   = {ev: g for ev, g in ab.groupby('event')}
evs_ab    = list(ev_g_ab.keys())
ev_gp_ab  = {ev: g['gross_pnl'].values for ev, g in ev_g_ab.items()}
ev_fee_ab = {ev: g['fee'].values        for ev, g in ev_g_ab.items()}

boot_gp_ab  = np.empty(BOOT_REPS)
boot_fee_ab = np.empty(BOOT_REPS)
for r in range(BOOT_REPS):
    samp = np.random.choice(evs_ab, size=len(evs_ab), replace=True)
    boot_gp_ab[r]  = np.concatenate([ev_gp_ab[e]  for e in samp]).mean()
    boot_fee_ab[r] = np.concatenate([ev_fee_ab[e] for e in samp]).mean()

ex_mean_gp   = float(ab['gross_pnl'].mean())
ex_mean_fee  = float(ab['fee'].mean())
ex_mean_ec   = float(ab['entry_cost'].mean())
ex_breakeven = ex_mean_gp - ex_mean_fee

boot_gaf = boot_gp_ab - boot_fee_ab
ci_gaf_lo, ci_gaf_hi = np.percentile(boot_gaf, [2.5, 97.5])
ex_c_sig = float(ci_gaf_lo)   # c where CI lower crosses 0

print('Exploitability:')
print(f'  N={len(ab)} candidates, {len(evs_ab)} events')
print(f'  Gross edge after fee (c=0): {ex_breakeven*100:.3f}c')
print(f'  95% CI at c=0            : [{ci_gaf_lo*100:.3f}c, {ci_gaf_hi*100:.3f}c]')
print(f'  Significance cutoff c    : {ex_c_sig*100:.3f}c')
print()

C_GRID  = [0.000, 0.005, 0.010, 0.015, 0.020]
ex_grid = []
for c in C_GRID:
    g = dict(c=c, net=ex_breakeven-c, lo=ci_gaf_lo-c, hi=ci_gaf_hi-c)
    g['sig'] = g['lo'] > 0
    ex_grid.append(g)
    print(f'  c={c*100:.1f}c: net={g["net"]*100:+.3f}c  CI=[{g["lo"]*100:+.3f}c,{g["hi"]*100:+.3f}c]  sig={g["sig"]}')

# ── B. Domain band stats ──────────────────────────────────────────────────
print()
print('Domain band stats:')

dom_stats = {}
for dom in ('Sports', 'Political'):
    dc = cand[cand['domain'] == dom]
    fl_sub = dc[dc['snapshot_price'] <= FLOOR_MAX]
    ls_sub = dc[(dc['snapshot_price'] > FLOOR_MAX) & (dc['snapshot_price'] < LS_MAX)]
    fv_sub = dc[dc['snapshot_price'] > FAV_MIN]

    dom_stats[dom] = {}
    for bname, sub in [('floored', fl_sub), ('longshot', ls_sub), ('favorite', fv_sub)]:
        mp  = float(sub['snapshot_price'].mean()) if len(sub) > 0 else float('nan')
        wr, lo, hi = _band_boot(sub)
        dom_stats[dom][bname] = dict(
            mp=mp, wr=wr, lo=lo, hi=hi,
            n=len(sub),
            n_ev=sub['event'].nunique() if len(sub) > 0 else 0,
            n_win=int(sub['won'].sum()) if len(sub) > 0 else 0
        )
        print(f'  {dom:<12} {bname:<10}: mp={mp:.4f}  wr={wr:.4f}  '
              f'CI=[{lo:.4f},{hi:.4f}]  n={len(sub)}')

print()
print('All computations done.')


Exploitability:
  N=369 candidates, 104 events
  Gross edge after fee (c=0): 3.507c
  95% CI at c=0            : [1.940c, 4.991c]
  Significance cutoff c    : 1.940c

  c=0.0c: net=+3.507c  CI=[+1.940c,+4.991c]  sig=True
  c=0.5c: net=+3.007c  CI=[+1.440c,+4.491c]  sig=True
  c=1.0c: net=+2.507c  CI=[+0.940c,+3.991c]  sig=True
  c=1.5c: net=+2.007c  CI=[+0.440c,+3.491c]  sig=True
  c=2.0c: net=+1.507c  CI=[-0.060c,+2.991c]  sig=False

Domain band stats:
  Sports       floored   : mp=0.0114  wr=0.0046  CI=[0.0000,0.0120]  n=659


  Sports       longshot  : mp=0.0596  wr=0.0146  CI=[0.0000,0.0326]  n=205
  Sports       favorite  : mp=0.7700  wr=0.8696  CI=[0.7391,1.0000]  n=23
  Political    floored   : mp=0.0118  wr=0.0090  CI=[0.0000,0.0198]  n=333


  Political    longshot  : mp=0.0707  wr=0.0387  CI=[0.0125,0.0710]  n=155
  Political    favorite  : mp=0.7972  wr=0.8289  CI=[0.7432,0.9054]  n=76

All computations done.


In [3]:
# ── Figure 1: Exploitability forest plot (slide-3 style) ──────────────────
C_GRID     = [0.000, 0.005, 0.010, 0.015, 0.020]
row_labels = ['c = 0.0¢', 'c = 0.5¢', 'c = 1.0¢', 'c = 1.5¢', 'c = 2.0¢']
y_pos      = list(range(len(C_GRID) - 1, -1, -1))   # 4..0, top→bottom

C_SIG  = '#E69F00'   # orange  — significant (matches slide-3 longshot color)
C_NS   = '#CCCCCC'   # grey    — not significant (matches slide-3 muted color)
C_NULL = '#444444'   # null line

FL = 20   # axis-label font  (matches slide-3)
FT = 15   # tick font
FA = 16   # row-label / value font

fig, ax = plt.subplots(figsize=(9, 5.5))

for c, label, y in zip(C_GRID, row_labels, y_pos):
    net = (ex_breakeven - c) * 100
    lo  = (ci_gaf_lo   - c) * 100
    hi  = (ci_gaf_hi   - c) * 100
    sig = lo > 0
    col = C_SIG if sig else C_NS

    # CI line
    ax.plot([lo, hi], [y, y], color=col, lw=2.5, solid_capstyle='round', zorder=3)
    # Caps
    cap_h = 0.13
    ax.plot([lo, lo], [y - cap_h, y + cap_h], color=col, lw=2.5, zorder=3)
    ax.plot([hi, hi], [y - cap_h, y + cap_h], color=col, lw=2.5, zorder=3)
    # Point estimate square
    ax.scatter([net], [y], marker='s', s=100, color=col,
               edgecolors='white', linewidths=1.4, zorder=4)

    # Row label — left
    ax.text(-1.3, y, label, va='center', ha='right', fontsize=FA, color='#333333')

    # Net edge value — right (just the number, colored by significance)
    ax.text(hi + 0.25, y,
            '{:+.2f}¢'.format(net),
            va='center', ha='left', fontsize=FA, color=col, fontweight='bold')

# Null reference line
ax.axvline(0, color=C_NULL, lw=1.8, ls='--', alpha=0.75, zorder=2)

ax.set_title('Longshot edge is erased under realistic transaction costs',
             loc='left', fontsize=FL - 2, pad=10, color='#222222')

ax.set_xlabel('Net edge per NO contract  (cents)', fontsize=FL, labelpad=10)
ax.tick_params(labelsize=FT)
ax.set_yticks([])
ax.set_xlim(-2.0, 8.5)
ax.set_ylim(-0.6, len(C_GRID) - 0.3)
ax.grid(axis='x', alpha=0.09, linewidth=0.7, color='grey')
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()
fp = os.path.join(OUT_DIR, 'fig_exploitability.pdf')
fig.savefig(fp, bbox_inches='tight')
print('Saved -> ' + fp)
plt.close(fig)

Saved -> /Users/josephsemelroth/Desktop/QSS 20 Final Project/output/fig_exploitability.pdf


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CALIBRATION REGRESSION FIGURE
# Above-floor sample (snapshot_price > FLOOR_MAX); pooled binned means;
# Model 1 OLS fit line; 45° perfect-calibration diagonal; house style.
# ══════════════════════════════════════════════════════════════════════════════

try:
    import statsmodels.formula.api as smf
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.run([_sys.executable, '-m', 'pip', 'install', '--quiet', 'statsmodels'], check=True)
    import statsmodels.formula.api as smf

np.random.seed(RNG_SEED)

# ── Above-floor sample + Model 1 fit ─────────────────────────────────────────
ab_reg = cand[cand['snapshot_price'] > FLOOR_MAX].copy().reset_index(drop=True)

m1_reg = smf.ols('won ~ snapshot_price', data=ab_reg).fit(
    cov_type='cluster', cov_kwds={'groups': ab_reg['event']}
)
alpha_r = m1_reg.params['Intercept']
beta_r  = m1_reg.params['snapshot_price']
print(f'Fitted: α = {alpha_r:.4f},  β = {beta_r:.4f}')

# ── Binned calibration stats (event-clustered CI) ─────────────────────────────
BINS_R = [(0.02, 0.04), (0.04, 0.07), (0.07, 0.15), (0.15, 0.50),
          (0.50, 0.70), (0.70, 0.80), (0.80, 0.90), (0.90, 1.00)]

def _bst(df):
    n, nw = len(df), int(df['won'].sum())
    if n == 0: return None
    mp, wr = float(df['snapshot_price'].mean()), float(df['won'].mean())
    if nw == 0:
        return dict(mp=mp, wr=wr, lo=0.0, hi=float(_cp_upper(n)))
    if nw == n:
        # All winners — bootstrap collapses to 1.0 every replicate.
        # Use Clopper-Pearson lower bound: Beta(0.025; n, 1) = 0.025^(1/n).
        cp_lo = float(0.025 ** (1.0 / n))
        return dict(mp=mp, wr=wr, lo=cp_lo, hi=1.0)
    ev_g = {ev: g['won'].values.astype(float) for ev, g in df.groupby('event')}
    evs  = list(ev_g.keys())
    boot = [np.concatenate([ev_g[e] for e in
                            np.random.choice(evs, len(evs), replace=True)]).mean()
            for _ in range(BOOT_REPS)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return dict(mp=mp, wr=wr, lo=lo, hi=hi)

rows_r = []
for lo_b, hi_b in BINS_R:
    mask = ((ab_reg['snapshot_price'] >  lo_b) &
            (ab_reg['snapshot_price'] <= hi_b if hi_b < 1.0
             else ab_reg['snapshot_price'] < 1.0))
    s = _bst(ab_reg[mask])
    if s: rows_r.append(s)

# ── Regression line ───────────────────────────────────────────────────────────
x_grid  = np.linspace(ab_reg['snapshot_price'].min(),
                       ab_reg['snapshot_price'].max(), 300)
y_hat_r = alpha_r + beta_r * x_grid

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# 45° perfect-calibration diagonal
ax.plot([0, 1], [0, 1], '--', color=C_DIA, lw=1.6, zorder=1, alpha=0.75)
ax.text(0.605, 0.542, 'perfect calibration', rotation=43,
        color=C_DIA, fontsize=FA - 3, alpha=0.65, style='italic',
        fontfamily='DejaVu Sans', zorder=3,
        bbox=dict(facecolor='white', edgecolor='none', pad=3))

# OLS fit line
ax.plot(x_grid, y_hat_r, '-', color=C_DIA, lw=2.2, zorder=4)

# Calibration points — pooled across domains, single color
C_DOT = '#2d6a9f'
MS = 10
for r in rows_r:
    ax.errorbar(r['mp'], r['wr'],
                yerr=[[max(0.0, r['wr'] - r['lo'])],
                       [max(0.0, r['hi'] - r['wr'])]],
                fmt='o', color=C_DOT, ecolor=C_DOT,
                markersize=MS, markeredgecolor='white', markeredgewidth=1.2,
                capsize=4, elinewidth=1.6, lw=1.6, zorder=5)

# Title
ax.set_title('Prices are steeper than perfect calibration',
             loc='left', fontsize=FL - 2, pad=10,
             fontfamily='DejaVu Sans', color='#222222')

# Axes
ax.set_xlabel('Market-implied probability  (snapshot price)',
              fontsize=FL, labelpad=10)
ax.set_ylabel('Actual win rate', fontsize=FL, labelpad=10)
ax.tick_params(labelsize=FT)
ax.set_xlim(-0.02, 1.04)
ax.set_ylim(-0.07, 1.10)
ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Spines — left + bottom only, light gray
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(alpha=0.09, color='grey', linewidth=0.7)

plt.tight_layout()

fp = os.path.join(OUT_DIR, 'fig_calibration_regression.pdf')
fig.savefig(fp, bbox_inches='tight', facecolor='white')
print(f'Saved -> {fp}')
plt.close(fig)

Fitted: α = -0.0598,  β = 1.1388


Saved -> /Users/josephsemelroth/Desktop/QSS 20 Final Project/output/fig_calibration_regression.pdf
